#### Purpose of this notebook is to determine if Python can fetch the Trap 104 chart page and does the returned HTML contain the chart data?

In [12]:
import requests
from pathlib import Path
import pandas as pd

In [2]:
url = "https://maps.eddmaps.org/line/sitedatabyyear/?aggregate=sum&sub=10378&project=1441&proj=1441&site=29224&showdate"

response = requests.get(url)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("HTML length:", len(response.text))

Status code: 403
Content-Type: text/html
HTML length: 919


Direct python request gives 403 (forbidden). We need to check whether the server is rejecting 
Python's default headers. 

Try a polite browser-like User-Agent.

The User-Agent header specifically identifies the client making the request.

In [ ]:
# Browser like request
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/125.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)

print("Status code:", response.status_code)
print("Content-Type:", response.headers.get("Content-Type"))
print("HTML length:", len(response.text))

Status code: 200
Content-Type: text/html;charset=UTF-8
HTML length: 16930


#### Search the returned HTML for key terms

In [ ]:
# !r tells the f‑string to format the value using repr(), which shows the Python 
# literal representation.

terms = ["echarts", "chart", "series", "2020", "2021", "29224", "Trap 104"]

for term in terms:
    print(f"Contains {term!r}:", term in response.text)

Contains 'echarts': True
Contains 'chart': True
Contains 'series': True
Contains '2020': False
Contains '2021': False
Contains '29224': True
Contains 'Trap 104': True


Python successfully fetched the Trap 104 chart page and the page clearly knows that 
'site_id = 29224' and 'trap_id = 104'

IMPORTANT: A kew-word search is a signal not proof. Seeing 'series == True' does 
not automatically means the data are present. It only means the word series appear 
somewhere in the HTML.

In [6]:
# Print the HTML around the word 'series'

text = response.text

idx = text.find('series')

print("Index of 'series':", idx)
print(text[idx - 500: idx + 1500])

Index of 'series': 7458
er: 'canvas',
            useDirtyRect: false
        });
        chart.showLoading();
        $.ajax({
            url: "data.cfm?_=292986.359697&aggregate=sum&sub=10378&end=2026&proj=1441&states=&UseDate=actual&siteid=29224",
            type: 'GET',
            dataType: 'json', 
            crossDomain: true,
            xhrFields: { withCredentials: true },
            success: function(data) {
                if(Array.isArray(data) && data.length > 0) {
                    let series = [];
                    for(let i = 0; i < data.length; i++) {
                        values = [];
                        for(j = 0; j < data[i].values.length; j++) {
                            values.push([data[i].values[j].label, data[i].values[j].value]);
                        }
                        s = {
                            name: data[i].id,
                            type:'line',
                            smooth:true,
                            data:

The HTML does not directly contain the full series data. The HTML contains 
JavaScript instructions that fetch the data from a separate JSON endpoint. 

$.ajax() means the page is making a background HTTP request.

dataType: 'json' means the response is expected to be structured JSON.

The page receives data, then builds the ECharts series manually.

#### Request the JSON endpoint directly

In [ ]:
# Can Python directly fetch the JSON data endpoint that the chart uses?

# Because the AJAX URL is relative, we need to attach it to the chart page’s 
# base path:

data_url = "https://maps.eddmaps.org/line/sitedatabyyear/data.cfm?_=292986.359697&aggregate=sum&sub=10378&end=2026&proj=1441&states=&UseDate=actual&siteid=29224"

data_response = requests.get(data_url, headers=headers)

print("Status code:", data_response.status_code)
print("Content-type:", data_response.headers.get('Content-Type'))
print("Response length:", len(data_response.text))
print(data_response.text[:500])

Status code: 200
Content-type: text/html;charset=UTF-8
Response length: 18340

[

	
	{
	"id" : "2020",
	"sub" : "2020",
	"values" : [
	{
				"value" : 0,
				"label" : "2024-04-22"
			}
			,{
				"value" : 0,
				"label" : "2024-04-28"
			}
			,{
				"value" : 0,
				"label" : "2024-05-06"
			}
			,{
				"value" : 0,
				"label" : "2024-05-15"
			}
			,{
				"value" : 0,
				"label" : "2024-05-22"
			}
			,{
				"value" : 0,
				"label" : "2024-05-28"
			}
			,{
				"value" : 0,
				"label" : "2024-06-12"
			}
			,{
				"value" : 0,
	


Each year object has- 

'id'       → true observation year

'values'   → list of weekly observations

Each value object has- 

'label'    → plotted date, normalized to 2024

'value'    → whitefly count

#### Parse the response as JSON

In [9]:
data = data_response.json()

print(type(data))
print("Number of year-series objects:", len(data))

print("First object keys:", data[0].keys())
print("First object id:", data[0]["id"])
print("Number of values in first object:", len(data[0]["values"]))
print("First 3 values:", data[0]["values"][:3])

<class 'list'>
Number of year-series objects: 7
First object keys: dict_keys(['id', 'sub', 'values'])
First object id: 2020
Number of values in first object: 30
First 3 values: [{'value': 0, 'label': '2024-04-22'}, {'value': 0, 'label': '2024-04-28'}, {'value': 0, 'label': '2024-05-06'}]


So, now we know for the Trap 104, the JSON has 7 year-series objects. 

The first object is: id = 2020; values = 30 weekly observations

#### Flatten only the 1st year for Trap 104

In [10]:
first_year = data[0]

year = first_year['id']
values = first_year['values']

rows = []

for point in values:
    plotted_date = point['label']
    whitefly_count = point['value']
    date_collected = year + '-' + plotted_date[5:]

    row = {
        'site_id': 29224,
        'trap_id': 104,
        'year': year,
        'plotted_date': plotted_date,
        'date_collected': date_collected,
        'whitefly_count': whitefly_count,
    }

    rows.append(row)

print("Number of rows created:", len(rows))
print("First 5 rows:")
rows[:5]

Number of rows created: 30
First 5 rows:


[{'site_id': 29224,
  'trap_id': 104,
  'year': '2020',
  'plotted_date': '2024-04-22',
  'date_collected': '2020-04-22',
  'whitefly_count': 0},
 {'site_id': 29224,
  'trap_id': 104,
  'year': '2020',
  'plotted_date': '2024-04-28',
  'date_collected': '2020-04-28',
  'whitefly_count': 0},
 {'site_id': 29224,
  'trap_id': 104,
  'year': '2020',
  'plotted_date': '2024-05-06',
  'date_collected': '2020-05-06',
  'whitefly_count': 0},
 {'site_id': 29224,
  'trap_id': 104,
  'year': '2020',
  'plotted_date': '2024-05-15',
  'date_collected': '2020-05-15',
  'whitefly_count': 0},
 {'site_id': 29224,
  'trap_id': 104,
  'year': '2020',
  'plotted_date': '2024-05-22',
  'date_collected': '2020-05-22',
  'whitefly_count': 0}]

#### Flatten all years for the Trap 104

In [11]:
rows = []

for year_series in data:
    year = year_series['id']

    for point in year_series['values']:
        plotted_date = point['label']
        whitefly_count = point['value']
        date_collected = year + '-' + plotted_date[5:]

        row = {
            "site_id": 29224,
            "trap_id": 104,
            "year": year,
            "plotted_date": plotted_date,
            "date_collected": date_collected,
            "whitefly_count": whitefly_count,
        }

        rows.append(row)

print("Total rows created:", len(rows))
print("First 3 rows:")
print(rows[:3])

print("Last 3 rows:")
print(rows[-3:])
    

Total rows created: 301
First 3 rows:
[{'site_id': 29224, 'trap_id': 104, 'year': '2020', 'plotted_date': '2024-04-22', 'date_collected': '2020-04-22', 'whitefly_count': 0}, {'site_id': 29224, 'trap_id': 104, 'year': '2020', 'plotted_date': '2024-04-28', 'date_collected': '2020-04-28', 'whitefly_count': 0}, {'site_id': 29224, 'trap_id': 104, 'year': '2020', 'plotted_date': '2024-05-06', 'date_collected': '2020-05-06', 'whitefly_count': 0}]
Last 3 rows:
[{'site_id': 29224, 'trap_id': 104, 'year': '2026', 'plotted_date': '2024-05-18', 'date_collected': '2026-05-18', 'whitefly_count': 0}, {'site_id': 29224, 'trap_id': 104, 'year': '2026', 'plotted_date': '2024-05-26', 'date_collected': '2026-05-26', 'whitefly_count': 0}, {'site_id': 29224, 'trap_id': 104, 'year': '2026', 'plotted_date': '2024-06-02', 'date_collected': '2026-06-02', 'whitefly_count': 0}]


#### Convert rows into pandas dataframe and inspect

In [ ]:
df = pd.DataFrame(rows)

print(df.shape)
df.head()

(301, 6)


,site_id,trap_id,year,plotted_date,date_collected,whitefly_count
0,29224,104,2020,2024-04-22,2020-04-22,0
1,29224,104,2020,2024-04-28,2020-04-28,0
2,29224,104,2020,2024-05-06,2020-05-06,0
3,29224,104,2020,2024-05-15,2020-05-15,0
4,29224,104,2020,2024-05-22,2020-05-22,0


In [14]:
df.tail()

,site_id,trap_id,year,plotted_date,date_collected,whitefly_count
296,29224,104,2026,2024-05-04,2026-05-04,0
297,29224,104,2026,2024-05-11,2026-05-11,0
298,29224,104,2026,2024-05-18,2026-05-18,0
299,29224,104,2026,2024-05-26,2026-05-26,0
300,29224,104,2026,2024-06-02,2026-06-02,0


In [15]:
df.groupby('year').size()

year
2020    30
2021    50
2022    49
2023    50
2024    49
2025    51
2026    22
dtype: int64

In [16]:
df['whitefly_count'].describe()

count     301.000000
mean       33.627907
std       148.763933
min         0.000000
25%         0.000000
50%         0.000000
75%         3.000000
max      2000.000000
Name: whitefly_count, dtype: float64

In [17]:
# Save the Trap 104 dataframe

interim_path = Path("../data/interim/trap_104_site_29224_weekly_counts.csv")

df.to_csv(interim_path, index=False)

print("Saved file to:", interim_path)

Saved file to: ../data/interim/trap_104_site_29224_weekly_counts.csv


## Single-trap extraction result

We successfully extracted weekly silverleaf whitefly trap-count data for Trap 104, corresponding to EDDMapS site_id 29224. The trap-specific chart page did not embed the full data directly in the HTML. Instead, the HTML contained a JavaScript AJAX request to a `data.cfm` endpoint. Requesting that endpoint with a browser-like User-Agent returned JSON-like data containing year-wise series objects.

Each year object contained an `id` field representing the true observation year and a `values` list containing plotted dates and whitefly counts. The plotted dates use 2024 as a normalized plotting year, so the actual collection date was reconstructed as:

`date_collected = year + "-" + plotted_date[5:]`

The final dataframe contained 301 rows, matching the manually inspected ECharts series counts:
2020 = 30, 2021 = 50, 2022 = 49, 2023 = 50, 2024 = 49, 2025 = 51, 2026 = 22.

The cleaned single-trap CSV was saved to:

`data/interim/trap_104_site_29224_weekly_counts.csv`

## Refactor the single-trap extraction into a reusable function

Now that Trap 104 has been successfully extracted and validated, the next goal is to move from one-off notebook code to a small reusable function. This is the first step toward scaling from one trap to many traps. The function will still focus on one site/trap at a time, but it will make the extraction logic easier to test on additional traps, such as Trap 105.

In [18]:
root_url = "https://maps.eddmaps.org/line/sitedatabyyear/data.cfm?_=292986.359697&aggregate=sum&sub=10378&end=2026&proj=1441&states=&UseDate=actual&siteid="

def extract_trap_weekly_counts(site_id, trap_id, headers):
    url = root_url +str(site_id)

    # request the endpoint with browser-like headers
    response = requests.get(url, headers=headers)

    # stop early if the request failed
    response.raise_for_status()

    # parse the JSON response
    data = response.json()

    rows = []
    for year_series in data:
        year = year_series['id']

        for point in year_series['values']:
            plotted_date = point['label']
            whitefly_count = point['value']
            date_collected = year + '-' + plotted_date[5:]

            row = {
                "site_id": site_id,
                "trap_id": trap_id,
                "year": year,
                "plotted_date": plotted_date,
                "date_collected": date_collected,
                "whitefly_count": whitefly_count,
            }

            rows.append(row)

    df = pd.DataFrame(rows)

    return df

In [19]:
# test the function
df_104 = extract_trap_weekly_counts(29224, 104, headers)
df_104.shape

(301, 6)

In [22]:
df_105 = extract_trap_weekly_counts(29225, 105, headers)
df_105.shape

(290, 6)

In [23]:
df_105.groupby("year").size()

year
2020    30
2021    47
2022    48
2023    47
2024    50
2025    47
2026    21
dtype: int64

Trap 105 returned 290 rows instead of the previously browser-observed 289 rows. The difference came from the 2026 series, which now contains 21 observations instead of 20. This likely reflects a newly added weekly monitoring record in the live endpoint rather than an extraction error.

### Inspect the map iframe to discover available trap/site IDs

In [24]:
map_url = (
    "https://maps.eddmaps.org/point/trap/relativecatch/index.cfm"
    "?sub=10378"
    "&proj=1441"
    "&project=1441"
    "&lat=31.35"
    "&lng=-83.60"
    "&zoom=10.5"
    "&map=136"
    "&shownodata=1"
    "&maxzoom=12"
    "&circlemarker"
    "&expiredata=14"
)

map_response = requests.get(map_url, headers=headers)

print("Status code:", map_response.status_code)
print("Content type:", map_response.headers.get("content-type"))
print("HTML length:", len(map_response.text))

Status code: 200
Content type: text/html;charset=UTF-8
HTML length: 25958


### Search the map HTML for useful terms

In [27]:
map_html = map_response.text

terms = [
    "site",
    "siteid",
    "29224",
    "29225",
    "Trap",
    "trap",
    "latitude",
    "longitude",
    "lat",
    "lng",
    "marker",
    "geojson",
    "data.cfm",
    "ajax",
]

for term in terms:
    print(f"{term!r}", term in map_html)

'site' True
'siteid' False
'29224' False
'29225' False
'Trap' False
'trap' True
'latitude' True
'longitude' True
'lat' True
'lng' True
'marker' True
'geojson' False
'data.cfm' False
'ajax' True


### Print snippets around term 'ajax'

In [29]:
def print_context(text, term, window=500):
    idx = text.find(term)
    
    if idx == -1:
        print(f"'{term}' not found")
    else:
        start = max(idx - window, 0)
        end = min(idx + window, len(text))
        print(text[start:end])

In [30]:
print_context(map_html, "ajax", window=1500)



<html lang="en">
    <head>
        <link rel="stylesheet" href="https://bugwoodcloud.org/CDN/openlayers/7.3/ol.css" type="text/css">
        <link rel="stylesheet" href="https://bugwoodcloud.org/CDN/openlayers-ext/4.0.7/dist/ol-ext.min.css" type="text/css">
        <link rel="stylesheet" href="https://cdnjs.cloudflare.com/ajax/libs/font-awesome/4.7.0/css/font-awesome.min.css">
       

        <script src="https://ajax.googleapis.com/ajax/libs/jquery/3.5.1/jquery.min.js"></script>
        <link rel="stylesheet" href="https://bugwoodcloud.org/CDN/bootstrap/3.1.1/css/bootstrap.min.css">
        <script src="https://bugwoodcloud.org/CDN/bootstrap/3.1.1/js/bootstrap.min.js"></script>

        <script src="https://bugwoodcloud.org/CDN/openlayers/7.3/dist/ol.js"></script>
        <script src="https://bugwoodcloud.org/CDN/openlayers-ext/4.0.7/dist/ol-ext.min.js"></script>
        <script type="text/javascript" src="https://bugwoodcloud.org/CDN/openlayers-ext/4.0.7/dist/extra/FontAwesomeDef

In [31]:
request_terms = [
    "$.ajax",
    "$.get",
    "$.post",
    "fetch(",
    ".load(",
    "url:",
    "source:",
]

for term in request_terms:
    print(term, map_html.find(term))

$.ajax 12511
$.get -1
$.post -1
fetch( -1
.load( -1
url: 12541
source: 11797


In [32]:
print_context(map_html, "$.ajax", window=1500)

Cmd + scroll to zoom the map</p></div>
        </div>

        <i class="fa fa-map-marker" title="fa-map-marker"></i>

        <script>
            var lat = '31.35';
            var lng = '-83.60';
            var zoom = '10.5';
            var maxzoom = '12'
        </script>
            

        <script src="/point/js/map.js?v=2"></script>
        <script>
            var params = null, points = {};
            /*Status categories*/
            colors = {'Low':'#4EB265','Low-Med':'#F7F056','Med-High':'#EE8026','High':'#DC050C','Inactive Site':'#777777','No Data':'#ffffff'};
            decodequerystring(); // initialize

            // Cluster Source
            var clusterSource=new ol.source.Cluster({
                distance: 10,
                source: new ol.source.Vector(),
                attributions: "EDDMapS."
            });
            // Animated cluster layer
            var clusterLayer = new ol.layer.AnimatedCluster({
                name: 'Cluster',
               

In [33]:
from urllib.parse import urlparse

parsed_map_url = urlparse(map_url)

print(parsed_map_url.query)

sub=10378&proj=1441&project=1441&lat=31.35&lng=-83.60&zoom=10.5&map=136&shownodata=1&maxzoom=12&circlemarker&expiredata=14


In [34]:
marker_data_url = (
    "https://maps.eddmaps.org/point/trap/relativecatch/data.cfc"
    "?method=relativecatch&"
    + parsed_map_url.query
)

marker_response = requests.get(marker_data_url, headers=headers)

print("Status code:", marker_response.status_code)
print("Content type:", marker_response.headers.get("content-type"))
print("Response length:", len(marker_response.text))
print(marker_response.text[:500])

Status code: 200
Content type: application/json;charset=UTF-8
Response length: 4074
{"records":[[29221,0,7,1,1,1,1,0.0,1,7,"June, 02 2026 00:00:00","Trap 101",31.46603,-83.465008,"Active","April, 22 2020 00:00:00"],[29222,0,7,1,1,1,1,0.0,1,7,"June, 02 2026 00:00:00","Trap 102",31.465087,-83.405028,"Active","April, 22 2020 00:00:00"],[29224,0,7,1,1,1,1,0.0,1,7,"June, 02 2026 00:00:00","Trap 104",31.557827,-83.480183,"Active","April, 22 2020 00:00:00"],[29225,0,7,1,1,1,1,0.0,1,7,"June, 02 2026 00:00:00","Trap 105",31.541213,-83.57304,"Active","April, 22 2020 00:00:00"],[29226,0,7


In [35]:
marker_data = marker_response.json()

print(type(marker_data))
print(marker_data.keys())
print(type(marker_data["records"]))
print(len(marker_data["records"]))

<class 'dict'>
dict_keys(['records', 'columns', 'count'])
<class 'list'>
31


In [36]:
marker_data["records"][0]

[29221,
 0,
 7,
 1,
 1,
 1,
 1,
 0.0,
 1,
 7,
 'June, 02 2026 00:00:00',
 'Trap 101',
 31.46603,
 -83.465008,
 'Active',
 'April, 22 2020 00:00:00']

In [37]:
len(marker_data["records"])

31

In [38]:
marker_data["records"][:5]

[[29221,
  0,
  7,
  1,
  1,
  1,
  1,
  0.0,
  1,
  7,
  'June, 02 2026 00:00:00',
  'Trap 101',
  31.46603,
  -83.465008,
  'Active',
  'April, 22 2020 00:00:00'],
 [29222,
  0,
  7,
  1,
  1,
  1,
  1,
  0.0,
  1,
  7,
  'June, 02 2026 00:00:00',
  'Trap 102',
  31.465087,
  -83.405028,
  'Active',
  'April, 22 2020 00:00:00'],
 [29224,
  0,
  7,
  1,
  1,
  1,
  1,
  0.0,
  1,
  7,
  'June, 02 2026 00:00:00',
  'Trap 104',
  31.557827,
  -83.480183,
  'Active',
  'April, 22 2020 00:00:00'],
 [29225,
  0,
  7,
  1,
  1,
  1,
  1,
  0.0,
  1,
  7,
  'June, 02 2026 00:00:00',
  'Trap 105',
  31.541213,
  -83.57304,
  'Active',
  'April, 22 2020 00:00:00'],
 [29226,
  0,
  7,
  1,
  1,
  1,
  1,
  0.0,
  1,
  7,
  'June, 02 2026 00:00:00',
  'Trap 106',
  31.524275,
  -83.662878,
  'Active',
  'April, 22 2020 00:00:00']]

In [39]:
marker_data["columns"]

['SITEID',
 'TOTALQTY',
 'CATCHPERIOD',
 'DENSERANK',
 'TRAPS',
 'QUARTILE',
 'RANKS',
 'PERCENTRANK',
 'CUSTOMRANK',
 'DAYSSINCE',
 'LASTREPORT',
 'SITENAME',
 'LAT',
 'LON',
 'STATUS',
 'FIRSTREPORT']

In [40]:
trap_metadata_raw = pd.DataFrame(
    marker_data["records"],
    columns=marker_data["columns"]
)

trap_metadata_raw.head()

,SITEID,TOTALQTY,CATCHPERIOD,DENSERANK,TRAPS,QUARTILE,RANKS,PERCENTRANK,CUSTOMRANK,DAYSSINCE,LASTREPORT,SITENAME,LAT,LON,STATUS,FIRSTREPORT
0,29221,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,7.0,"June, 02 2026 00:00:00",Trap 101,31.466030,-83.465008,Active,"April, 22 2020 00:00:00"
1,29222,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,7.0,"June, 02 2026 00:00:00",Trap 102,31.465087,-83.405028,Active,"April, 22 2020 00:00:00"
2,29224,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,7.0,"June, 02 2026 00:00:00",Trap 104,31.557827,-83.480183,Active,"April, 22 2020 00:00:00"
3,29225,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,7.0,"June, 02 2026 00:00:00",Trap 105,31.541213,-83.573040,Active,"April, 22 2020 00:00:00"
4,29226,0.0,7.0,1.0,1.0,1.0,1.0,0.0,1.0,7.0,"June, 02 2026 00:00:00",Trap 106,31.524275,-83.662878,Active,"April, 22 2020 00:00:00"


In [41]:
trap_metadata_raw.shape

(31, 16)

In [42]:
trap_metadata_raw[["SITEID", "SITENAME", "LAT", "LON", "STATUS", "FIRSTREPORT", "LASTREPORT"]].head()

,SITEID,SITENAME,LAT,LON,STATUS,FIRSTREPORT,LASTREPORT
0,29221,Trap 101,31.466030,-83.465008,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
1,29222,Trap 102,31.465087,-83.405028,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
2,29224,Trap 104,31.557827,-83.480183,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
3,29225,Trap 105,31.541213,-83.573040,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
4,29226,Trap 106,31.524275,-83.662878,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"


In [43]:
trap_metadata = trap_metadata_raw[
    ["SITEID", "SITENAME", "LAT", "LON", "STATUS", "FIRSTREPORT", "LASTREPORT"]
].copy()

trap_metadata = trap_metadata.rename(columns={
    "SITEID": "site_id",
    "SITENAME": "trap_label",
    "LAT": "latitude",
    "LON": "longitude",
    "STATUS": "status",
    "FIRSTREPORT": "first_report",
    "LASTREPORT": "last_report",
})

trap_metadata.head()

,site_id,trap_label,latitude,longitude,status,first_report,last_report
0,29221,Trap 101,31.466030,-83.465008,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
1,29222,Trap 102,31.465087,-83.405028,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
2,29224,Trap 104,31.557827,-83.480183,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
3,29225,Trap 105,31.541213,-83.573040,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"
4,29226,Trap 106,31.524275,-83.662878,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00"


In [45]:
trap_metadata["trap_label"].tolist()

['Trap 101',
 'Trap 102',
 'Trap 104',
 'Trap 105',
 'Trap 106',
 'Trap 107',
 'Trap 108',
 'Trap 109',
 'Trap 110',
 'Trap 111',
 'Trap 112',
 'Trap 113',
 'Trap 114',
 'Trap 115',
 'Trap 116',
 'Trap 117',
 'Trap 118',
 'Trap 119',
 'Trap 120',
 'Kendallwood',
 'Bass rd',
 'Hwy 319',
 'Funston/Sale city',
 'Mike Horne Rd',
 'Old Berlin ',
 'Shiver Rd.',
 'Melton Brinson Rd.',
 'HWY 200 ',
 'Chafin Rd',
 'Morven Field',
 'Dixie Field']

In [49]:
trap_metadata["trap_id"] = (
    trap_metadata["trap_label"]
    .str.extract(r"^Trap\s+(\d+)$")[0]
    .astype("Int64")
)

trap_metadata.tail(10)

,site_id,trap_label,latitude,longitude,status,first_report,last_report,trap_id
21,31141,Hwy 319,31.032005,-83.857199,OldData,"February, 14 2025 00:00:00","March, 27 2026 00:00:00",<NA>
22,31142,Funston/Sale city,31.237018,-83.911690,OldData,"February, 14 2025 00:00:00","March, 27 2026 00:00:00",<NA>
23,31144,Mike Horne Rd,31.279251,-83.892223,OldData,"February, 14 2025 00:00:00","March, 27 2026 00:00:00",<NA>
24,31147,Old Berlin,31.087651,-83.708702,OldData,"February, 14 2025 00:00:00","March, 27 2026 00:00:00",<NA>
25,31565,Shiver Rd.,31.001615,-84.215862,OldData,"April, 23 2025 00:00:00",NaN,<NA>
26,31566,Melton Brinson Rd.,30.867297,-84.359697,OldData,"April, 23 2025 00:00:00",NaN,<NA>
27,31627,HWY 200,31.370427,-84.895381,Active,"May, 01 2025 00:00:00","June, 08 2026 00:00:00",<NA>
28,32085,Chafin Rd,31.218826,-83.617481,OldData,"June, 27 2025 00:00:00","March, 27 2026 00:00:00",<NA>
29,32409,Morven Field,30.933030,-83.479394,OldData,"August, 21 2025 00:00:00",NaN,<NA>
30,32410,Dixie Field,30.798771,-83.688641,OldData,"August, 21 2025 00:00:00",NaN,<NA>


In [50]:
trap_metadata["status"].value_counts(dropna=False)

status
Active     19
OldData    12
Name: count, dtype: int64

In [51]:
trap_metadata[["site_id", "trap_label", "status", "first_report", "last_report", "trap_id"]]

,site_id,trap_label,status,first_report,last_report,trap_id
0,29221,Trap 101,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",101
1,29222,Trap 102,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",102
2,29224,Trap 104,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",104
3,29225,Trap 105,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",105
4,29226,Trap 106,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",106
5,29227,Trap 107,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",107
6,29228,Trap 108,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",108
7,29229,Trap 109,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",109
8,29230,Trap 110,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",110
9,29231,Trap 111,Active,"April, 22 2020 00:00:00","June, 02 2026 00:00:00",111


In [52]:
trap_metadata["first_report"] = pd.to_datetime(
    trap_metadata["first_report"],
    errors="coerce"
)

trap_metadata["last_report"] = pd.to_datetime(
    trap_metadata["last_report"],
    errors="coerce"
)

trap_metadata[[
    "site_id",
    "trap_label",
    "status",
    "first_report",
    "last_report",
    "trap_id"
]].tail(10)

,site_id,trap_label,status,first_report,last_report,trap_id
21,31141,Hwy 319,OldData,2025-02-14,2026-03-27,<NA>
22,31142,Funston/Sale city,OldData,2025-02-14,2026-03-27,<NA>
23,31144,Mike Horne Rd,OldData,2025-02-14,2026-03-27,<NA>
24,31147,Old Berlin,OldData,2025-02-14,2026-03-27,<NA>
25,31565,Shiver Rd.,OldData,2025-04-23,NaT,<NA>
26,31566,Melton Brinson Rd.,OldData,2025-04-23,NaT,<NA>
27,31627,HWY 200,Active,2025-05-01,2026-06-08,<NA>
28,32085,Chafin Rd,OldData,2025-06-27,2026-03-27,<NA>
29,32409,Morven Field,OldData,2025-08-21,NaT,<NA>
30,32410,Dixie Field,OldData,2025-08-21,NaT,<NA>


In [53]:
trap_metadata.isna().sum()

site_id          0
trap_label       0
latitude         0
longitude        0
status           0
first_report     0
last_report      4
trap_id         12
dtype: int64

In [54]:
trap_metadata.info()

<class 'pandas.DataFrame'>
RangeIndex: 31 entries, 0 to 30
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   site_id       31 non-null     int64         
 1   trap_label    31 non-null     str           
 2   latitude      31 non-null     float64       
 3   longitude     31 non-null     float64       
 4   status        31 non-null     str           
 5   first_report  31 non-null     datetime64[us]
 6   last_report   27 non-null     datetime64[us]
 7   trap_id       19 non-null     Int64         
dtypes: Int64(1), datetime64[us](2), float64(2), int64(1), str(2)
memory usage: 2.1 KB


In [55]:
metadata_output_path = "../data/interim/trap_metadata.csv"

trap_metadata.to_csv(metadata_output_path, index=False)

print("Saved metadata to:", metadata_output_path)

Saved metadata to: ../data/interim/trap_metadata.csv


In [56]:
trap_metadata_check = pd.read_csv(metadata_output_path)

trap_metadata_check.head()
trap_metadata_check.shape

(31, 8)